# Next day: the return and the win, draw or loss, with LightGBM, a GRU and an MLP

The method of [`docs/ml_plan.md`](../docs/ml_plan.md) run end to end, on
[`src/tools/ml_models/`](../src/tools/ml_models/). Two models, each forecasting one thing about
tomorrow (the owner's choice):

- **the return model**: tomorrow's return as a point forecast (the median) and its 68% and 90%
  bands;
- **the win/draw/loss model**: the probabilities that tomorrow is a win (return above +0.2%), a
  draw (within ±0.2%) or a loss (below −0.2%).

Each comes as a LightGBM, as a GRU and as a scikit-learn MLP. **LightGBM and the GRUs are
loaded from their latest saved versions, not retrained**, when this computer has them (an earlier
run saved them in `models/models`); only what is missing is trained. The MLP, the new method, is
trained, checked and saved, then compared with them. Bars, features, labels, folds, baselines,
metrics, the models and the sanity checks all come from the package, as the tests check them.

| Section | Question |
|---|---|
| 1 Bars, labels, features | What does a model see at the close of day t, and what must it forecast for t+1? Does any feature look ahead? |
| 2 Walk-forward | How are the folds cut (6-month steps from 2023), and what is purged at their boundaries? |
| 3 onward | **Switched off** (commented out) while the workflow is rebuilt step by step. |
| 3 Baselines | What does each model have to beat? |
| 4 LightGBM | The two models as gradient-boosted trees, loaded from their latest saved version. |
| 5 GRU | The two models from a 60-day window, three input sets, loaded likewise. |
| 6 MLP | The two models as scikit-learn multilayer perceptrons, trained here, and a lead for tuning them. |
| 7 Results and sanity checks | The usual metrics first (R², MAE, RMSE, IC and hit rate for the return; accuracy, F1, MCC, log loss and confusion matrices for W/D/L), then the plan's checks. On the synthetic tickers: W/D/L no better than the volatility baseline but better than the class frequencies; the point forecast no better than the EWMA median; the range better than a constant band. |
| 8 The checks can fail | A planted signal must be found; two leaks must be caught; a forecast must not change when later bars are removed. |
| 9 Save the final models | Each model trained in this run, trained once more on every labelled row and saved with its out-of-sample forecasts under a unique id. |
| 10 The MLP against the saved models | The MLP's out-of-sample losses against each saved LightGBM and GRU, per ticker, without retraining them. |

**Data.** The default is two synthetic tickers, SYN-INDEX and SYN-GOLD (generated, offline, not
market data). To run
on real prices, set `DATA = 'local'` and list the symbols in `LOCAL_TICKERS`. Then choose with
`DOWNLOAD`: `True` downloads them from Yahoo and saves them to the git-ignored `data/local` (or
brings the saved files up to date), `False` uses the bars already saved there. **Do not commit
this notebook with outputs**: run on saved data, its tables and charts would be market data, and
a test refuses a committed ML notebook with outputs. Clear all outputs before committing.

**Needs** the `stats` and `ml` extras: `pip install -e ".[stats,ml]"` (for a CPU-only PyTorch,
first `pip install torch --index-url https://download.pytorch.org/whl/cpu`). While sections 3
onward are switched off, a run takes about a minute.

In [ ]:
# ── Environment setup: Google Colab vs local ────────────────────────
# Locally the project is installed once from the repo root with `pip install -e ".[stats,ml]"`,
# so `from src.tools...` resolves from any working directory. On Colab, an editable install is
# only seen by a new interpreter, so the repo root goes on sys.path for this kernel.
import os
import sys

REPO = '/content/drive/MyDrive/github/quant_dev'

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')
    if not os.path.isdir(REPO):
        raise FileNotFoundError(f'No repo at {REPO}; edit REPO to point at your clone.')
    %pip install -q -e "{REPO}[stats,ml]"
    sys.path.insert(0, REPO)

In [ ]:
import os
import time
from functools import partial

import numpy as np
import pandas as pd
from IPython.display import display

from src.tools import ml_models as ml
from src.tools.price_return import local_tickers, save_all

try:
    import lightgbm
    import sklearn
    import torch
except ImportError as exc:
    raise ImportError('This notebook needs the ml extra: pip install -e ".[stats,ml]"') from exc

print(f'LightGBM {lightgbm.__version__}, PyTorch {torch.__version__}, scikit-learn {sklearn.__version__}')

## Parameters

The design choices confirmed in the plan are the package's defaults: return quantiles at
`ml.TAUS` and a win/draw/loss threshold of 0.2%. The folds are cut by calendar date: the first
training set ends on `FIRST_TRAIN_END`, and each fold tests the next `FOLD_MONTHS` months
(section 2). `LONG` are the tickers the sanity checks judge: both of them.

In [ ]:
DATA = 'synthetic'                       # 'synthetic' (offline) or 'local' (real Yahoo bars in data/local)
LOCAL_TICKERS = ['ES=F', 'NQ=F', 'GC=F', 'CL=F']    # the symbols when DATA == 'local'
DOWNLOAD = False                         # with 'local': True downloads from Yahoo and saves to data/local
                                         # (or updates the saved files); False uses the bars already saved
START_DATE = '2016-01-01'                # with DOWNLOAD: where a symbol's first download starts
SYNTHETIC_TICKERS = ['SYN-INDEX', 'SYN-GOLD']     # the synthetic tickers this notebook uses
TICKERS = SYNTHETIC_TICKERS if DATA == 'synthetic' else LOCAL_TICKERS
LONG = list(TICKERS)                     # judged by the sanity checks
WDL_THRESHOLD = ml.WDL_THRESHOLD         # 0.002: a win above +0.2%, a loss below -0.2%
N_JOBS = None                            # tickers in parallel; None is one per CPU
GRU_TICKERS = TICKERS                    # the GRUs are the slow part; shorten this list to save time
GRU_INPUTS = list(ml.GRU_INPUTS)         # 'returns_range' (the default), 'returns', 'features'
GRU_STEP = 252                           # retrain every 252 rows; the plan's 63 takes four times as long
FIRST_TRAIN_END = '2022-12-31'           # section 2: the first training set ends here
FOLD_MONTHS = 6                          # section 2: each fold tests the next 6 months, then joins training
MIN_TEST_MONTHS = 3                      # section 2: a shorter last test window joins the one before
REUSE_SAVED = True                       # load the latest saved LightGBM and GRUs; train only what is missing
SAVE_MODELS = True                       # section 9 writes models/models (git-ignored) and model_parameters
MLP_SWEEP = True                         # section 6: a short regularization sweep on the first ticker

### The data: download and save, or load what is saved

With `DATA = 'local'` the bars come from `data/local`, which only ever holds what was downloaded
from Yahoo (git-ignored: it is market data). With `DOWNLOAD = True` each symbol is downloaded and
saved first:

- a symbol not yet saved is downloaded from `START_DATE`;
- a saved one is brought up to date from a few days before its last bar, up to yesterday (today's
  bar is not final);
- bars Yahoo has revised since are replaced, and listed below.

With `DOWNLOAD = False` nothing is downloaded, and every symbol must already be saved. Either way,
the table shows what the notebook will use. The synthetic tickers are generated here and need
neither.

After new bars arrive, the LightGBM and GRU versions loaded in sections 4 and 5 are still the
ones saved earlier, trained up to their own last bar. Set `REUSE_SAVED = False` to retrain them
on the new bars.

In [ ]:
if DATA == 'local':
    if DOWNLOAD:
        result = save_all(TICKERS, start_date=START_DATE,
                          on_result=lambda e: print(f"{e['symbol']}: " + (
                              f"failed: {e['error']}" if 'error' in e else
                              f"{'downloaded' if e['created'] else 'updated'}, {e['added']} bars added, "
                              f"{len(e['revised'])} revised")))
        for e in result['saved']:
            if e['revised']:
                print(f"{e['symbol']}: bars Yahoo revised since they were saved:")
                display(pd.DataFrame(e['revised']))
        for e in result['failed']:
            print(f"{e['symbol']}: the download failed, so its saved file, if any, is used as it was")
    saved = {t['symbol']: t for t in local_tickers()}
    missing = [t for t in TICKERS if t not in saved]
    if missing:
        raise FileNotFoundError(
            f'Not saved in data/local: {missing}. Set DOWNLOAD = True to download them from Yahoo, '
            f'or take them out of LOCAL_TICKERS.')
    display(pd.DataFrame([saved[t] for t in TICKERS]).set_index('symbol')
            .rename(columns={'updated': 'last downloaded'}))
else:
    print(f'Synthetic tickers, generated here: {", ".join(TICKERS)}')

## 1 Bars, labels, features

`C_t` is the close of bar t. The only forward-looking columns are the two labels: tomorrow's
return `y_ret = r_{t+1}` and tomorrow's outcome `y_wdl`: 1 (win) above +0.2%, 0 (draw) within
±0.2%, −1 (loss) below −0.2%. A return that spans a non-positive close (SYN-OIL on 2020-04-20) is
dropped, as `price_return` drops it.

The features (`ml.FEATURES`) use bar t and the bars before it, and none depends on the price
level. Two describe the recent wins and losses: `wdl`, today's outcome against the same
threshold, and `streak`, the run of wins (+) or losses (−) ending today, up to 10 either way,
reset to 0 by a draw.

In [ ]:
BARS = {t: ml.ticker_bars(t, source=DATA) for t in TICKERS}
DATA_BY_TICKER = {t: ml.dataset(b, WDL_THRESHOLD) for t, b in BARS.items()}
pd.DataFrame({t: {'rows': len(d), 'from': d.index[0].date(), 'to': d.index[-1].date(),
                  'wins': f"{(d['y_wdl'] == 1).mean():.1%}", 'draws': f"{(d['y_wdl'] == 0).mean():.1%}",
                  'losses': f"{(d['y_wdl'] == -1).mean():.1%}",
                  'longest streak': f"{d['streak'].min():+.0f} / {d['streak'].max():+.0f}"}
              for t, d in DATA_BY_TICKER.items()}).T

In [ ]:
DATA_BY_TICKER[TICKERS[0]][['ret_0', 'wdl', 'streak', 'y_ret', 'y_wdl']].tail(10)

**No look-ahead.** The test `ta_tools` uses for its indicators: cut the bars after t, recompute,
and every feature at or before t must be unchanged. A feature that peeked at a later bar would
change. (`tests/test_ml_models.py` runs this per feature; here it is shown once.)

In [ ]:
def check_no_lookahead(bars, cuts=(300, 1180, 2000)):
    full = ml.features(bars, WDL_THRESHOLD)
    for cut in cuts:
        pd.testing.assert_frame_equal(ml.features(bars.iloc[:cut + 1], WDL_THRESHOLD), full.iloc[:cut + 1],
                                      check_exact=False, rtol=1e-12)
    return f'{len(ml.FEATURES)} features unchanged at {len(cuts)} cut points'

check_no_lookahead(BARS[TICKERS[0]])

## 2 Walk-forward: 6-month steps

Every test is a forecast made only with what was known at the time. The data are split by
calendar date, in 6-month steps:

1. **The first training set** is every day up to 31 December 2022 (from 2016, after the 60-day
   warm-up the features need). **Its test set** is the next 6 months, January to June 2023.
2. **Then those 6 months join the training set**, and the next 6 months are the test set. And so
   on, to the end of the data.
3. **A test set has at least 3 months of data.** The last one runs to the end of the data; if
   that leaves it shorter than `MIN_TEST_MONTHS`, it is not a fold of its own, and its days join
   the test set before it. For example, with data to August 2026, the fold training to 30 June
   2026 would test only July and part of August. So the last fold trains to 31 December 2025 and
   tests from 1 January 2026 to the end of the data. A window whose data reaches within a week
   of the 3-month mark counts as long enough: the last day before a weekend or holiday, and the
   last bar (which has no label yet), fall a few days short.

So there are only a few folds, each with a long training history and half a year of test days.
Together the test sets cover every day from 2023 exactly once. A ticker whose history starts
late starts testing once it has at least a year of training rows.

A label at day t looks at day t+1 (h = 1). Had it looked h days ahead, the last h−1 training
days' labels would reach into the test set, so those days would be dropped (purged) from
training. For the next-day labels, none are.

In [ ]:
FOLDS = {t: ml.walk_forward_dates(DATA_BY_TICKER[t].index, FIRST_TRAIN_END, FOLD_MONTHS,
                                  min_test_months=MIN_TEST_MONTHS)
         for t in TICKERS}
display(pd.DataFrame({t: {'folds': len(f), 'first test day': DATA_BY_TICKER[t].index[f[0][1][0]].date() if f else None,
                          'last test day': DATA_BY_TICKER[t].index[f[-1][1][-1]].date() if f else None,
                          'days tested': sum(len(test) for _, test in f)}
                      for t, f in FOLDS.items()}).T)
t = TICKERS[0]
ml.fold_table(DATA_BY_TICKER[t].index, FOLDS[t]).style.set_caption(f'{t}: the folds')

In [ ]:
# The purge: next-day labels (h = 1) need none; a 5-day label would drop 4 training days.
d = DATA_BY_TICKER[TICKERS[0]]
for h in (1, 5):
    train, test = ml.walk_forward_dates(d.index, FIRST_TRAIN_END, FOLD_MONTHS, horizon=h,
                                        min_test_months=MIN_TEST_MONTHS)[0]
    print(f'h = {h}: last training day {d.index[train[-1]]:%Y-%m-%d}, its label reaches '
          f'{d.index[train[-1] + h]:%Y-%m-%d}, first test day {d.index[test[0]]:%Y-%m-%d}: '
          f'{test[0] - train[-1] - 1} days purged')

### The folds on the whole history

The close over the complete data, with every fold's test window marked, and **the last fold
highlighted**: its training set (blue) and its test set (orange). Then every feature and label
over the same history, with the same highlight. Discrete columns are shown as the share of days
at each value in that training set and that test set, and `wdl` and `streak` also over time.
Drag across any time panel to zoom; the others follow.

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

t = TICKERS[0]
d, bars, folds = DATA_BY_TICKER[t], BARS[t], FOLDS[t]
train, test = folds[-1]                       # the last fold
fig = go.Figure(go.Scatter(x=bars.index, y=bars['Close'], mode='lines', name='close',
                           line=dict(width=1.2, color='#0b0b0b'),
                           hovertemplate='%{x|%Y-%m-%d}: %{y:,.2f}<extra></extra>'))
fig.add_vrect(x0=d.index[train[0]], x1=d.index[train[-1]], fillcolor='#2a78d6', opacity=0.10,
              line_width=0, layer='below', annotation_text='last fold: training set',
              annotation_position='top left')
fig.add_vrect(x0=d.index[test[0]], x1=d.index[test[-1]], fillcolor='#eb6834', opacity=0.25,
              line_width=0, layer='below', annotation_text='test set', annotation_position='top right')
for k, (_, test_k) in enumerate(folds, start=1):
    fig.add_vline(x=d.index[test_k[0]], line=dict(color='#898781', width=1, dash='dot'))
    fig.add_annotation(x=d.index[test_k[0]], y=0, yref='paper', text=f'{k}', showarrow=False,
                       xanchor='left', yanchor='bottom', font=dict(size=10, color='#52514e'))
fig.update_layout(template='plotly_white', height=420, margin=dict(t=60, l=50, r=20, b=40),
                  title=f'{t}: the complete data; dotted lines start each fold\'s test set (numbered)',
                  yaxis_title='close', showlegend=False)
fig.show()

In [ ]:
LABELS = ['y_ret', 'y_wdl']
discrete = ['dow', 'wdl', 'streak', 'y_wdl']
panels = []                                   # (title, column, kind)
for c in ml.FEATURES + LABELS:
    panels.append((c, c, 'bars' if c in discrete else 'line'))
    if c == 'wdl':
        panels.append(('wdl over time (dots: each day; line: 20-day mean)', c, 'wdl_time'))
    if c == 'streak':
        panels.append(('streak over time', c, 'streak_time'))
n_cols = 3
n_rows = -(-len(panels) // n_cols)
fig = make_subplots(rows=n_rows, cols=n_cols, subplot_titles=[p[0] for p in panels],
                    vertical_spacing=0.03, horizontal_spacing=0.06)
for k, (title, c, kind) in enumerate(panels):
    row, col = k // n_cols + 1, k % n_cols + 1
    color = '#eb6834' if c in LABELS else '#2a78d6'
    if kind == 'bars':
        for rows_, name, shade in ((train, 'last fold: training set', '#2a78d6'),
                                   (test, 'last fold: test set', '#eb6834')):
            share = d[c].iloc[rows_].value_counts(normalize=True).sort_index()
            fig.add_trace(go.Bar(x=share.index, y=share, name=name, marker_color=shade,
                                 legendgroup=name, showlegend=c == discrete[0],
                                 hovertemplate=f'{c} = %{{x}}: %{{y:.1%}} of days<extra>{name}</extra>'),
                          row=row, col=col)
        fig.update_yaxes(tickformat='.0%', row=row, col=col)
        fig.update_xaxes(dtick=1, row=row, col=col)
        continue
    if kind == 'wdl_time':
        fig.add_trace(go.Scattergl(x=d.index, y=d[c], mode='markers', name='wdl', showlegend=False,
                                   marker=dict(size=3, color=color, opacity=0.35),
                                   hovertemplate='wdl: %{y:.0f}<extra>%{x|%Y-%m-%d}</extra>'),
                      row=row, col=col)
        fig.add_trace(go.Scattergl(x=d.index, y=d[c].rolling(20).mean(), mode='lines',
                                   name='wdl, 20-day mean', showlegend=False,
                                   line=dict(width=2, color='#0b0b0b'),
                                   hovertemplate='20-day mean: %{y:+.2f}<extra>%{x|%Y-%m-%d}</extra>'),
                      row=row, col=col)
        fig.update_yaxes(tickvals=[-1, 0, 1], ticktext=['loss', 'draw', 'win'], row=row, col=col)
    else:                                     # 'line' and 'streak_time'
        fig.add_trace(go.Scattergl(x=d.index, y=d[c], mode='lines', name=c, showlegend=False,
                                   line=dict(width=1, color=color,
                                             shape='hv' if kind == 'streak_time' else 'linear'),
                                   hovertemplate=f'{c}: %{{y:.4g}}<extra>%{{x|%Y-%m-%d}}</extra>'),
                      row=row, col=col)
    fig.update_xaxes(matches='x', row=row, col=col)           # every time panel zooms together
    fig.add_vrect(x0=d.index[train[0]], x1=d.index[train[-1]], fillcolor='#2a78d6',
                  opacity=0.08, line_width=0, layer='below', row=row, col=col)
    fig.add_vrect(x0=d.index[test[0]], x1=d.index[test[-1]], fillcolor='#eb6834',
                  opacity=0.25, line_width=0, layer='below', row=row, col=col)
fig.update_layout(height=170 * n_rows, template='plotly_white', barmode='group',
                  margin=dict(t=90, l=40, r=20, b=30),
                  legend=dict(orientation='h', yanchor='bottom', y=1.01, x=0),
                  title=f'{t}: every feature and label, the last fold highlighted (labels in orange)')
fig.update_annotations(font_size=11)
fig.show()

## Sections 3 onward: switched off

Everything below is commented out while the workflow is rebuilt one step at a time, so a run
stops here. That includes the saved-model tools (listing, loading, and `models_for`, which loads a
saved model or trains one), since no models are saved now. The code is kept: to bring a cell back, select its contents and toggle the comments
(Ctrl+/ in Colab and Jupyter). Note that the cells below still use the old 63-day folds; they
will be moved onto `FOLDS` as each step is rebuilt.

### The saved models available

`ml.list_models()` reads every model's parameters in `models/model_parameters` (tracked), and
`saved_here` says whether the model itself is in `models/models` on this computer (git-ignored,
so only where it was saved). The first table counts them per model and target for this run's
data and threshold. The second shows the version `REUSE_SAVED` will load for each ticker: the
newest on this computer. To load any one of them, `ml.load_model(model_id)`; to load the newest
version, `ml.load_latest(model, target, ticker, source=DATA)`.

In [ ]:
# Switched off: select all and toggle comments (Ctrl+/) to run this cell.
# AVAILABLE = ml.list_models()
# mine = AVAILABLE[(AVAILABLE['source'] == DATA) & (AVAILABLE['wdl_threshold'] == WDL_THRESHOLD)]
# print(f'{len(AVAILABLE)} saved models in {ml.PARAMETERS_DIR}, {int(AVAILABLE["saved_here"].sum())} '
#       f'of them on this computer; {len(mine)} for {DATA} data at a {WDL_THRESHOLD:.1%} threshold')
# if len(mine):
#     display(mine.groupby(['model', 'target']).agg(
#         tickers=('ticker', 'nunique'), versions=('spec_id', 'nunique'), models=('ticker', 'size'),
#         on_this_computer=('saved_here', 'sum'), newest=('created', 'max'), last_bar=('last_bar', 'max')))
#     newest = (mine[mine['saved_here'] & mine['ticker'].isin(TICKERS)].sort_values('created')
#               .reset_index().groupby(['model', 'target', 'ticker']).tail(1)
#               .set_index(['model', 'target', 'ticker']).sort_index())
#     display(newest[['model_id', 'created', 'last_bar', 'test_days', 'loss']]
#             .style.set_caption('The version REUSE_SAVED loads for each ticker'))
# else:
#     print('None yet: sections 4 to 6 train every model, and section 9 saves them.')

In [ ]:
# Switched off: select all and toggle comments (Ctrl+/) to run this cell.
# # One saved model in detail: here the newest on this computer. Any model_id above works.
# if len(mine) and mine['saved_here'].any():
#     example = ml.load_model(mine[mine['saved_here']].sort_values('created').index[-1])
#     p = example.parameters
#     settings = {**p['model']['settings'], 'features': f"{len(p['model']['features'])} features"}
#     print(f"{example.model_id}\n  {p['model']['class']}, settings {settings}\n"
#           f"  trained on {p['data']['ticker']} ({p['data']['source']}) from {p['data']['first']} to "
#           f"{p['data']['last']}; walk-forward {p['walk_forward']['test_days']} test days, "
#           f"from {p['walk_forward']['test_from']} to {p['walk_forward']['test_to']}\n"
#           f"  saved {p['created']}, code as today: {p['model']['code'] == ml.code_hash(type(example.model))}")
#     display(example.forecast_next().to_frame('next day'))

### Load the saved models, or train them

`models_for` gives a model's walk-forward forecasts for each ticker. With `REUSE_SAVED` it loads
the **latest saved version** (`ml.load_latest`): the newest one on this computer for the same
model, target, ticker, data source and threshold. Its settings or code may be older, which is
what makes it a version, and the table says when a version's code differs from today's. Only
tickers with nothing saved are trained, and they are saved in section 9. `VERSIONS` records which
version each model is.

In [ ]:
# Switched off: select all and toggle comments (Ctrl+/) to run this cell.
# VERSIONS = []      # which version of each model this run uses
# TRAINED = []       # (ticker, model, forecasts, step): trained in this run, saved in section 9
# SAVED = {}         # (model, target, ticker): the saved model, loaded or (section 9) new
#
#
# def models_for(name, target, make_model, tickers, step=63, reuse=None):
#     """`name`'s walk-forward forecasts for each ticker: its latest saved version when reusing and
#     one is saved here, else trained now with `make_model()` (and saved in section 9)."""
#     reuse = REUSE_SAVED if reuse is None else reuse
#     out = {}
#     for t in (tickers if reuse else []):
#         saved = ml.load_latest(name, target, t, source=DATA, wdl_threshold=WDL_THRESHOLD)
#         if saved is not None:
#             out[t], SAVED[(name, target, t)] = saved.forecasts, saved
#             VERSIONS.append({'model': name, 'target': target, 'ticker': t, 'how': 'loaded',
#                              'model_id': saved.model_id, 'created': saved.parameters['created'],
#                              'code as today': saved.parameters['model']['code']
#                              == ml.code_hash(type(saved.model))})
#     missing = [t for t in tickers if t not in out]
#     if missing:
#         for fc in ml.evaluate_many(missing, make_model(), source=DATA, wdl_threshold=WDL_THRESHOLD,
#                                    n_jobs=N_JOBS, step=step):
#             out[fc.ticker] = fc
#             TRAINED.append((fc.ticker, make_model(), fc, step))
#             VERSIONS.append({'model': name, 'target': target, 'ticker': fc.ticker,
#                              'how': 'trained now', 'model_id': None, 'created': None,
#                              'code as today': True})
#     return {t: out[t] for t in tickers}
#
#
# def versions(name):
#     v = pd.DataFrame(VERSIONS)
#     return v[v['model'] == name].set_index(['target', 'ticker'])[['how', 'model_id', 'created',
#                                                                   'code as today']]

## 3 Baselines

Each is refitted per fold from the training rows, or uses only what is known at t.

**Win/draw/loss.** The training window's class frequencies; and a **volatility baseline**: the
chance that tomorrow clears the threshold given today's EWMA volatility (the training window's
standardized returns, scaled by today's volatility). With a fixed threshold, draws are commoner in
calm spells, so this baseline knows how likely a draw is, and nothing about direction.

**Return.** A constant-width band (the training quantiles), `price_range`'s lognormal ±zσ band on
the 250-day volatility, a 20-day σ band, and the EWMA volatility times the training quantiles of
`r_{t+1} / σ_t`. Each band's median is its point forecast.

In [ ]:
# Switched off: select all and toggle comments (Ctrl+/) to run this cell.
# rows = []
# for t in LONG:
#     d = DATA_BY_TICKER[t]
#     folds = ml.walk_forward(len(d))
#     wdl = ml.wdl_baselines(d, folds)
#     y = d.loc[wdl.index, 'y_wdl'].to_numpy()
#     row = {'ticker': t}
#     for b in ml.WDL_BASELINES:
#         row[f'W/D/L log loss, {b}'] = ml.multiclass_log_loss(wdl[b].to_numpy(), y, ml.WDL_CLASSES).mean()
#     bands = ml.range_baselines(d, folds)
#     yr = d.loc[bands.index, 'y_ret']
#     for band in ('constant', 'ewma_std_q'):
#         s, _ = ml.range_scores({tau: bands[(band, tau)] for tau in ml.TAUS}, yr)
#         row[f'pinball (bp), {band}'] = s['pinball'] * 1e4
#         row[f'cover 68%, {band}'] = s['cover_68']
#     rows.append(row)
# pd.DataFrame(rows).set_index('ticker').round(4)

## 4 LightGBM

`ml.LightGBMWDLModel`: one `multiclass` model. `ml.LightGBMReturnModel`: one `quantile` model per
τ on `r_{t+1} / σ_t`, scaled back by σ_t; the median is the point forecast. Both: 15 leaves, at
least 50 rows a leaf, learning rate 0.03, up to 300 trees, early-stopped on the purged tail and
refitted on the whole training window, deterministic on one thread.

**Loaded, not retrained.** Each ticker's latest saved version is loaded with its out-of-sample
forecasts (`models_for`, above). Only a ticker with none saved on this computer is trained now,
and saved in section 9.

In [ ]:
# Switched off: select all and toggle comments (Ctrl+/) to run this cell.
# t0 = time.time()
# LGB_W = models_for('LightGBM', 'wdl', ml.LightGBMWDLModel, TICKERS)
# LGB_R = models_for('LightGBM', 'return', ml.LightGBMReturnModel, TICKERS)
# print(f'LightGBM, both models, {len(TICKERS)} tickers: {time.time() - t0:.0f} s')
# display(versions('LightGBM'))
# pd.DataFrame({t: {'folds': len(LGB_W[t].folds),
#                   'W/D/L: median trees': int(LGB_W[t].folds['trees'].median()),
#                   'W/D/L: top features': ', '.join(ml.feature_importance(LGB_W[t]).index[:3]),
#                   'return median: top features': ', '.join(ml.feature_importance(LGB_R[t]).index[:3])}
#               for t in TICKERS}).T

In [ ]:
# Switched off: select all and toggle comments (Ctrl+/) to run this cell.
# t = TICKERS[0]
# ml.plot_folds(LGB_W[t]).show()
# ml.plot_feature_importance(ml.feature_importance(LGB_W[t]),
#                            title=f'{t}: W/D/L model, share of gain per feature').show()

## 5 GRU

`ml.GRUWDLModel` and `ml.GRUReturnModel`: one GRU layer of 32 units over the last 60 rows of its
inputs, standardized with the training window's means and standard deviations. The W/D/L network
ends in three logits (cross-entropy); the return network in five quantiles of the standardized
return, scaled back by σ_t and **recalibrated**: each τ is shifted by the τ-quantile of the
network's own errors on the validation tail, which keeps its bands from being too narrow. Early
stopping on the purged tail, seeded per fold, deterministic on CPU.

**Three input sets, kept until a model is trained on real data** (the owner's choice):
`'returns_range'` (each day's return and its true range, the provisional default), `'returns'`
(the return alone) and `'features'` (all the features LightGBM reads). They retrain every
`GRU_STEP` rows, over the same test days as LightGBM. **Loaded, not retrained**, as LightGBM.

In [ ]:
# Switched off: select all and toggle comments (Ctrl+/) to run this cell.
# GRU_W, GRU_R = {}, {}
# t0 = time.time()
# for inputs in GRU_INPUTS:
#     name = f'GRU ({inputs})'
#     GRU_W[inputs] = models_for(name, 'wdl', partial(ml.GRUWDLModel, features=inputs, threads=1),
#                                GRU_TICKERS, step=GRU_STEP)
#     GRU_R[inputs] = models_for(name, 'return', partial(ml.GRUReturnModel, features=inputs, threads=1),
#                                GRU_TICKERS, step=GRU_STEP)
#     print(f'{name}, both models: {time.time() - t0:.0f} s')
# pd.concat({f'GRU ({i})': versions(f'GRU ({i})') for i in GRU_INPUTS})

## 6 MLP (scikit-learn)

The new method, always trained here. `ml.MLPWDLModel` and `ml.MLPReturnModel` at the owner's
starting point: **two hidden layers, each as wide as the feature list (18 neurons), ReLU, Adam**,
and scikit-learn's defaults for everything else (`alpha` 0.0001, batches of 200, learning rate
0.001, at most 200 epochs, no early stopping).

- **W/D/L:** an `MLPClassifier` with those settings, since a regressor gives no probabilities. It
  trains on all the training rows.
- **Return:** an `MLPRegressor` on `r_{t+1} / σ_t`, whose forecast is the centre. It trains on the
  training rows less the purged last 20%. Its errors on that held-out tail set each τ's offset,
  as the GRU's recalibration does. The median is the point forecast; the offsets give the bands.
- **Two departures from the defaults,** which the plan's rules need:
  - a fixed seed per fold, since the default draws a new one each run;
  - inputs standardized on the training rows, since the MLP does not scale them.

Forecasts are computed from the trained weights, one row at a time (the same numbers as
scikit-learn's `predict`), so a saved model needs only its weights.

In [ ]:
# Switched off: select all and toggle comments (Ctrl+/) to run this cell.
# t0 = time.time()
# MLP_W = models_for('MLP', 'wdl', ml.MLPWDLModel, TICKERS, reuse=False)
# MLP_R = models_for('MLP', 'return', ml.MLPReturnModel, TICKERS, reuse=False)
# print(f'MLP, both models, {len(TICKERS)} tickers: {time.time() - t0:.0f} s')
# pd.DataFrame({t: {'folds': len(MLP_W[t].folds),
#                   'W/D/L: epochs (median)': int(MLP_W[t].folds['n_iter'].median()),
#                   'W/D/L: folds converged': f"{MLP_W[t].folds['converged'].mean():.0%}",
#                   'return: epochs (median)': int(MLP_R[t].folds['n_iter'].median()),
#                   'return: folds converged': f"{MLP_R[t].folds['converged'].mean():.0%}"}
#               for t in TICKERS}).T

"Converged" is scikit-learn's own test: the training loss stopped improving by its tolerance
before the last epoch. At 200 epochs it rarely has, and it says so with a warning, recorded per
fold here rather than printed.

### A lead for tuning: regularization

At the starting configuration the networks fit the noise, and section 7 shows what that costs.
Here are the same models on the first ticker with more regularization (`alpha`), or with
scikit-learn's early stopping, passed through `params`. Above zero is better than the baseline.
Set `MLP_SWEEP = False` to skip this (about two minutes).

In [ ]:
# Switched off: select all and toggle comments (Ctrl+/) to run this cell.
# if MLP_SWEEP:
#     t0 = time.time()
#     sweep, d = {}, DATA_BY_TICKER[TICKERS[0]]
#     for label, params in [('starting point', {}), ('early_stopping=True', {'early_stopping': True}),
#                           ('alpha=1', {'alpha': 1.0}), ('alpha=10', {'alpha': 10.0})]:
#         w = ml.walk_forward_forecasts(d, ml.MLPWDLModel(params=params), ticker=TICKERS[0])
#         r = ml.walk_forward_forecasts(d, ml.MLPReturnModel(params=params), ticker=TICKERS[0])
#         rt = ml.range_table(r)
#         sweep[label] = {'W/D/L log loss, gain over the volatility baseline': ml.wdl_scores(w)['gain_vs_volatility'],
#                         'point MAE, gain over the EWMA median (bp)': ml.point_scores(r)['gain_vs_ewma'] * 1e4,
#                         'range pinball, gain over the constant band':
#                             1 - rt.loc['model', 'pinball'] / rt.loc['constant', 'pinball']}
#     print(f'sweep: {time.time() - t0:.0f} s')
#     display(pd.DataFrame(sweep).T.style.format({
#         'W/D/L log loss, gain over the volatility baseline': '{:+.4f}',
#         'point MAE, gain over the EWMA median (bp)': '{:+.2f}',
#         'range pinball, gain over the constant band': '{:+.1%}'}).set_caption(f'{TICKERS[0]}: the MLP with more regularization'))

## 7 Results and sanity checks

On the synthetic tickers the sign of tomorrow's return cannot be forecast, but its volatility
can. So, for every method:

- **win/draw/loss**: the model must be no better than the volatility baseline (its 99% interval
  of the log-loss gain must not lie wholly above zero, and the gain must be at most 0.015), but
  it should beat the plain class frequencies, pooled over the tickers. The margin is there because
  the volatility baseline (EWMA) is not the best volatility forecast: the generator's own GARCH
  volatility beats it by up to 0.014 on SYN-INDEX, so a model may land a little above it;
- **the point forecast** (the median) must be no better than the EWMA band's median;
- **the range** must have a lower pinball loss than the constant band on each long ticker, misses
  that cluster less, and a pooled gain whose 95% interval lies wholly above zero.

LightGBM's and the GRUs' tables come from their saved forecasts, so they are the numbers of the
version loaded.

### Model metrics at a glance

The usual scores first: if a model is poor here, the detailed checks below will not rescue it.
Each table pools the long tickers' out-of-sample days, one row per model, with the baselines
underneath for scale.

**Return model (regression)**, on its point forecast (the median of tomorrow's return), with
errors in basis points of return:

- **R2**, as scikit-learn computes it (against the mean of the days scored), and **R2 vs no
  change** (against a forecast of zero: "the price will not change"). Above zero beats that
  benchmark. For daily returns a value near zero is normal, even for a good model; below zero is
  worse than the benchmark.
- **MAE, RMSE** and **bias** (above zero: forecasts too high).
- **IC**, the rank correlation of forecast and outcome, and its Pearson **corr**. An IC of a few
  hundredths is already useful for daily returns.
- **Hit rate**, the share of days with the sign right, beside **forecast up**, the share of days
  forecast up. A model that always says "up" has a hit rate equal to the share of up days.
- For the range: **68% and 90% coverage** (should be near 68% and 90%), the 68% band's
  **width**, and the mean **pinball** loss.

All on returns, not prices. Tomorrow's close is so close to today's that an R2 on price levels
is above 0.99 for any forecast, the no-change one included, so it says nothing.

**W/D/L model (classification)**, with the most probable outcome as each day's call:

- **accuracy**; compare it with the class frequencies' row, which calls the commonest outcome;
- **balanced accuracy** (1/3 is chance), **macro F1**, **weighted F1** and **MCC** (0 is
  chance), which do not reward always calling the commonest outcome. Pooled over tickers, even
  the baselines score above chance: knowing which ticker is calm (SYN-FX's commonest outcome is
  a draw, the others' a win) is worth something. So read every model against the baseline rows;
- **log loss** and **Brier**, on the probabilities themselves. These are what the models are
  trained on, and the fairest ranking.

Then the per-outcome report (precision, recall, F1, support) and the confusion matrices. Each
matrix comes **hard** (days counted under their most probable outcome) and **soft** (the
average probability given to each outcome, by what happened). Read the soft one down a column:
a model that tells draws apart gives a draw more probability on the draw row.

In [ ]:
# Switched off: select all and toggle comments (Ctrl+/) to run this cell.
# RETURN_MODELS = {'LightGBM': LGB_R, **{f'GRU ({i})': GRU_R[i] for i in GRU_INPUTS}, 'MLP': MLP_R}
# WDL_MODELS = {'LightGBM': LGB_W, **{f'GRU ({i})': GRU_W[i] for i in GRU_INPUTS}, 'MLP': MLP_W}
#
#
# def pooled(forecasts):
#     return [forecasts[t] for t in LONG if t in forecasts]
#
#
# rows = {name: ml.return_metrics(pooled(R)).loc['model'] for name, R in RETURN_MODELS.items()}
# base = ml.return_metrics(pooled(LGB_R))            # the baselines, on LightGBM's folds
# for band, label in (('no change', 'no change (zero)'), ('ewma_std_q', 'EWMA band (baseline)'),
#                     ('constant', 'constant band (baseline)')):
#     rows[label] = base.loc[band]
# RETURN_METRICS = pd.DataFrame(rows).T
# display(RETURN_METRICS.style.format(
#     {'days': '{:,.0f}', 'r2': '{:+.4f}', 'r2_vs_no_change': '{:+.4f}', 'mae_bp': '{:.2f}',
#      'rmse_bp': '{:.2f}', 'bias_bp': '{:+.2f}', 'corr': '{:+.3f}', 'ic': '{:+.3f}',
#      'hit_rate': '{:.1%}', 'forecast_up': '{:.1%}', 'cover_68': '{:.1%}', 'cover_90': '{:.1%}',
#      'width_68_bp': '{:.1f}', 'pinball_bp': '{:.2f}'}, na_rep='')
#     .set_caption(f'Return model: regression metrics, {len(LONG)} long tickers pooled (bp: basis points)'))
#
# per_ticker = pd.DataFrame({name: {t: ml.return_metrics(R[t]).loc['model', 'r2_vs_no_change']
#                                   for t in LONG if t in R} for name, R in RETURN_MODELS.items()})
# display(per_ticker.style.format('{:+.4f}').set_caption('R2 vs no change, per ticker'))

In [ ]:
# Switched off: select all and toggle comments (Ctrl+/) to run this cell.
# rows, gains = {}, {}
# for name, W in WDL_MODELS.items():
#     table, gain = ml.wdl_metrics(pooled(W))
#     rows[name] = table.loc['model']
#     gains[name] = gain.xs('volatility', level='over').stack()
# base, _ = ml.wdl_metrics(pooled(LGB_W))            # the baselines, on LightGBM's folds
# rows['volatility baseline'] = base.loc['volatility']
# rows['class frequencies (commonest outcome)'] = base.loc['frequencies']
# WDL_METRICS = pd.DataFrame(rows).T
# display(WDL_METRICS.style.format(
#     {'days': '{:,.0f}', **{c: '{:.1%}' for c in ('accuracy', 'balanced_accuracy', 'called_loss',
#                                                   'called_draw', 'called_win')},
#      **{c: '{:.3f}' for c in ('macro_f1', 'weighted_f1')}, 'mcc': '{:+.3f}',
#      'log_loss': '{:.4f}', 'brier': '{:.4f}'})
#     .set_caption(f'W/D/L model: classification metrics, {len(LONG)} long tickers pooled'))
# WDL_GAINS = pd.DataFrame(gains).T
# display(WDL_GAINS.style.format('{:+.4f}').set_caption(
#     'Gain over the volatility baseline, with 95% intervals: balanced accuracy, and log loss '
#     '(above zero: the model is better)'))

In [ ]:
# Switched off: select all and toggle comments (Ctrl+/) to run this cell.
# display(pd.concat({name: ml.wdl_class_report(pooled(W)) for name, W in WDL_MODELS.items()})
#         .style.format({'precision': '{:.3f}', 'recall': '{:.3f}', 'f1': '{:.3f}',
#                        'support': '{:,.0f}', 'called': '{:.1%}'})
#         .set_caption('Per outcome: precision, recall, F1, the days it happened, the share called'))
# for name, W in WDL_MODELS.items():
#     ml.plot_wdl_confusion(pooled(W)).show()
# ml.plot_wdl_confusion(pooled(LGB_W), soft=True).show()

In [ ]:
# Switched off: select all and toggle comments (Ctrl+/) to run this cell.
# WDL_COLS = ['days', 'share_draw', 'log_loss_frequencies', 'log_loss_volatility', 'log_loss_model',
#             'gain_vs_frequencies', 'gain_vs_volatility', 'lower_vs_volatility', 'upper_vs_volatility',
#             'no_better_than_volatility']
#
#
# def results(name, wdl, ret):
#     out = {}
#     if wdl:
#         w = pd.DataFrame([ml.wdl_scores(fc) for fc in wdl.values()])[WDL_COLS]
#         display(w.style.format({c: '{:+.4f}' if c.startswith(('gain', 'lower', 'upper')) else '{:.4f}'
#                                 for c in WDL_COLS[2:-1]} | {'share_draw': '{:.1%}'})
#                 .set_caption(f'{name}: win/draw/loss'))
#         _, pooled, above = ml.wdl_check([wdl[t] for t in LONG if t in wdl], 'frequencies', 'model')
#         chance = bool(w.loc[w.index.isin(LONG), 'no_better_than_volatility'].all())
#         print(f'{name}: W/D/L no better than the volatility baseline on every long ticker: {chance}; '
#               f'pooled gain over the class frequencies {pooled["estimate"]:+.4f} '
#               f'(95% interval {pooled["lower"]:+.4f} to {pooled["upper"]:+.4f})')
#         out['W/D/L no better than volatility'] = chance
#         out['W/D/L beats the frequencies'] = above
#     if ret:
#         p = pd.DataFrame([ml.point_scores(fc) for fc in ret.values()])
#         display((p.drop(columns='no_better_than_ewma') * 1e4).round(3).assign(ok=p['no_better_than_ewma'])
#                 .style.set_caption(f'{name}: point forecast, mean absolute error (bp)'))
#         ranges = pd.concat({t: ml.range_table(fc).loc[['constant', 'ewma_std_q', 'model'],
#                                                      ['cover_68', 'cover_90', 'kupiec_p_68',
#                                                       'christoffersen_p_68', 'cluster_ratio_68', 'pinball']]
#                             for t, fc in ret.items()})
#         ranges['pinball'] *= 1e4
#         display(ranges.style.format({'cover_68': '{:.1%}', 'cover_90': '{:.1%}', 'kupiec_p_68': '{:.3f}',
#                                      'christoffersen_p_68': '{:.2g}', 'cluster_ratio_68': '{:.2f}',
#                                      'pinball': '{:.2f}'}).set_caption(f'{name}: range (pinball in bp)'))
#         _, pooled, passes = ml.range_check([ret[t] for t in LONG if t in ret])
#         point = bool(p.loc[p.index.isin(LONG), 'no_better_than_ewma'].all())
#         print(f'{name}: point forecast no better than the EWMA median: {point}; range pooled gain '
#               f'{pooled["estimate"]:.1%} (95% interval {pooled["lower"]:.1%} to {pooled["upper"]:.1%}); '
#               f'range checks pass: {passes}')
#         out['point no better than EWMA median'] = point
#         out['range beats the constant band'] = passes
#         out['range pooled gain'] = f'{pooled["estimate"]:.1%} ({pooled["lower"]:.1%} to {pooled["upper"]:.1%})'
#     return out
#
#
# SUMMARY = {'LightGBM': results('LightGBM', LGB_W, LGB_R)}

In [ ]:
# Switched off: select all and toggle comments (Ctrl+/) to run this cell.
# for inputs in GRU_W:
#     SUMMARY[f'GRU ({inputs})'] = results(f'GRU ({inputs})', GRU_W[inputs], GRU_R[inputs])

In [ ]:
# Switched off: select all and toggle comments (Ctrl+/) to run this cell.
# SUMMARY['MLP'] = results('MLP', MLP_W, MLP_R)

### Charts

Reliability of the W/D/L forecasts (one curve per outcome; marker area grows with the days in
the bin), the return model's point forecast and bands through SYN-INDEX's scheduled sell-off, the
rolling coverage of its 68% interval, and every band's pinball loss against the constant one: for
LightGBM, then the same for the MLP.

In [ ]:
# Switched off: select all and toggle comments (Ctrl+/) to run this cell.
# t = TICKERS[0]
# ml.plot_wdl_reliability(LGB_W[t]).show()
# ml.plot_wdl_reliability(LGB_W[t], source='volatility').show()
# ml.plot_forecast_bands(LGB_R[t], BARS[t], '2020-02-01', '2020-04-30').show()
# ml.plot_coverage(LGB_R[t]).show()
# ml.plot_range_comparison([LGB_R[t] for t in LONG]).show()

In [ ]:
# Switched off: select all and toggle comments (Ctrl+/) to run this cell.
# ml.plot_wdl_reliability(MLP_W[t]).show()
# ml.plot_forecast_bands(MLP_R[t], BARS[t], '2020-02-01', '2020-04-30').show()
# ml.plot_range_comparison([MLP_R[t] for t in LONG]).show()

## 8 The checks can fail

A check that cannot fail proves nothing; a model that always forecast the volatility baseline
would pass trivially. So controls on each W/D/L model trained in this run:

1. **A planted signal.** SYN-INDEX with each day's sign made to repeat the last 65% of the time
   (sizes and volatility kept). The W/D/L model must now beat the volatility baseline.
2. **A leaked future return.** `r_{t+1}` added as a feature. The check must fail.
3. **A centred window.** A 5-day average centred on t, a realistic look-ahead bug: it carries
   `r_{t+1}` and `r_{t+2}`. The check must fail.
4. **A forecast cannot see past its day.** Remove every bar after a test date (keeping the next
   close, which is only that date's label), refit that fold, and both models' forecasts must be
   identical.

The MLP gets all four. LightGBM and the GRUs loaded from saved versions passed theirs when they
were trained (an earlier run of this notebook, and the package's tests), so they are not
retrained here; any trained in this run get theirs.

In [ ]:
# Switched off: select all and toggle comments (Ctrl+/) to run this cell.
# index_data = ml.dataset(ml.ticker_bars('SYN-INDEX'), WDL_THRESHOLD)
# r = ml.bar_returns(ml.ticker_bars('SYN-INDEX'))
# centred = index_data.assign(leak=r.rolling(5, center=True).mean().reindex(index_data.index)).dropna()
# centred.attrs = index_data.attrs
# planted = ml.dataset(ml.planted_signal_bars(), WDL_THRESHOLD)
# CONTROL_COLS = ['days', 'log_loss_volatility', 'log_loss_model', 'gain_vs_volatility',
#                 'lower_vs_volatility', 'upper_vs_volatility', 'no_better_than_volatility',
#                 'beats_volatility']
#
#
# def trained_now(prefix):
#     return any(v['how'] == 'trained now' and v['model'].startswith(prefix) for v in VERSIONS)
#
#
# def controls(make, leaky):
#     runs = {'planted signal': ml.walk_forward_forecasts(planted, make()),
#             'leaked r(t+1)': ml.walk_forward_forecasts(index_data.assign(leak=index_data['y_ret']), leaky),
#             'centred window': ml.walk_forward_forecasts(centred, leaky)}
#     out = pd.DataFrame({k: ml.wdl_scores(fc) for k, fc in runs.items()}).T[CONTROL_COLS]
#     out['must'] = ['beat the volatility baseline', 'fail the check', 'fail the check']
#     out['passes'] = [bool(out.loc['planted signal', 'beats_volatility']),
#                      not out.loc['leaked r(t+1)', 'no_better_than_volatility'],
#                      not out.loc['centred window', 'no_better_than_volatility']]
#     return out
#
#
# CONTROLS = {'MLP': controls(ml.MLPWDLModel, ml.MLPWDLModel(features=ml.FEATURES + ['leak']))}
# if trained_now('LightGBM'):
#     CONTROLS['LightGBM'] = controls(ml.LightGBMWDLModel,
#                                     ml.LightGBMWDLModel(features=ml.FEATURES + ['leak'], max_trees=60))
# pd.concat(CONTROLS)

In [ ]:
# Switched off: select all and toggle comments (Ctrl+/) to run this cell.
# # The planted signal for each GRU input set trained in this run.
# GRU_PLANTED = pd.DataFrame({
#     f'GRU ({inputs})': ml.wdl_scores(ml.walk_forward_forecasts(
#         planted, ml.GRUWDLModel(features=inputs), step=GRU_STEP))
#     for inputs in GRU_INPUTS if trained_now(f'GRU ({inputs})')}).T
# if len(GRU_PLANTED):
#     display(GRU_PLANTED[['days', 'log_loss_volatility', 'log_loss_model', 'gain_vs_volatility',
#                          'lower_vs_volatility', 'beats_volatility']])
# else:
#     print('Every GRU was loaded from a saved version: its planted-signal control ran when it was trained.')

In [ ]:
# Switched off: select all and toggle comments (Ctrl+/) to run this cell.
# bars = ml.ticker_bars('SYN-INDEX')
# checks = [('MLP', ml.MLPWDLModel, ml.MLPReturnModel, MLP_W, MLP_R)]
# if trained_now('LightGBM'):
#     checks.append(('LightGBM', ml.LightGBMWDLModel, ml.LightGBMReturnModel, LGB_W, LGB_R))
# UNCHANGED = {}
# for name, make_w, make_r, W, R in checks:
#     if 'SYN-INDEX' not in W:
#         continue                          # run on saved data: the check needs SYN-INDEX's forecasts
#     for position in (900, 1700, 2400):
#         date = index_data.index[position]
#         d = ml.dataset(bars.iloc[:bars.index.get_loc(date) + 2], WDL_THRESHOLD)
#         fold = next(f for f in ml.walk_forward(len(d)) if date in d.index[f[1]])
#         same_w = ml.walk_forward_forecasts(d, make_w(), folds=[fold]).wdl.loc[date, 'model'].equals(
#             W['SYN-INDEX'].wdl.loc[date, 'model'])
#         same_r = ml.walk_forward_forecasts(d, make_r(), folds=[fold]).bands.loc[date, 'model'].equals(
#             R['SYN-INDEX'].bands.loc[date, 'model'])
#         UNCHANGED[(name, f'{date:%Y-%m-%d}')] = same_w and same_r
# pd.Series(UNCHANGED, name='identical with the later bars removed', dtype=bool)

## Summary

At its starting configuration the MLP overfits the noise. So it passes the "no better than"
checks, which guard against look-ahead, by being worse than the baselines, and it fails the range
check and the planted signal. That is a configuration to tune, not a leak: section 6's sweep
shows every score moving towards the baselines as the regularization grows, and the leaks are
still caught.

In [ ]:
# Switched off: select all and toggle comments (Ctrl+/) to run this cell.
# display(pd.DataFrame(SUMMARY))
# for name, table in CONTROLS.items():
#     print(f'Controls ({name} W/D/L): ' + '; '.join(
#         f"{k}: {'passes' if ok else 'FAILS'}" for k, ok in table['passes'].items()))
# for name, row in GRU_PLANTED.iterrows():
#     print(f"{name}: planted signal found: {bool(row['beats_volatility'])} (gain {row['gain_vs_volatility']:+.4f})")
# if UNCHANGED:
#     print(f'Forecasts unchanged when later bars are removed: {all(UNCHANGED.values())}')

## 9 Save the final models

Every model trained in this run (the MLP, and any LightGBM or GRU that had no saved version) is
trained once more on every labelled row (`ml.finalize`, the same fit with all rows in the training
window) and saved:

- `models/models/<id>.pkl`, **git-ignored**: one object per model with everything in it: the
  parameters, the trained trees or weights, and the walk-forward out-of-sample forecasts. Those
  forecasts are what a later model is compared with, as section 10 does: a final model cannot be
  scored on the days it was trained on.
- `models/model_parameters/<id>.json`, **tracked**: the methodology (model, settings, features,
  threshold, walk-forward, a hash of the model code), the bars it was trained on (range and
  fingerprint), the final fit and summary scores. No per-day values, no weights.

**The id**, `{model}_{target}_{ticker}_{last bar}_{hash}` (e.g.
`mlp_wdl_SYN-INDEX_20260930_1c2d3e4f`): the hash covers the methodology and the bars, so the
same model on the same bars always gets the same id and is not saved twice, while a changed
setting, an edited model, or newer bars get a new id. `spec_id` hashes the methodology alone and
is shared by every ticker trained the same way. The loaded models are already saved; the table
lists the version of every model this run used.

In [ ]:
# Switched off: select all and toggle comments (Ctrl+/) to run this cell.
# t0 = time.time()
# NEW = []
# for step in sorted({s for *_, s in TRAINED}):
#     NEW += ml.finalize_many([(t, m, fc) for t, m, fc, s in TRAINED if s == step], n_jobs=N_JOBS,
#                             source=DATA, wdl_threshold=WDL_THRESHOLD, step=step)
# print(f'{len(NEW)} final models: {time.time() - t0:.0f} s')
# for s in NEW:
#     SAVED[(s.model.name, s.target, s.ticker)] = s
#     if SAVE_MODELS:
#         ml.save_model(s)
# if SAVE_MODELS and NEW:
#     print(f'saved in {ml.MODELS_DIR} and {ml.PARAMETERS_DIR}')
# for v in VERSIONS:
#     if v['how'] == 'trained now':
#         s = SAVED[(v['model'], v['target'], v['ticker'])]
#         v.update(model_id=s.model_id, created=s.parameters['created'])
# pd.DataFrame(VERSIONS).set_index(['model', 'target', 'ticker'])

### Load a model back and forecast the next day

`ml.load_model(id)` returns the saved object, ready to forecast; `ml.load_latest(model, target,
ticker)` the newest saved version. `forecast_next()` applies it to the ticker's latest bar (the
one with no label yet), so after saving newer bars the same model forecasts the newer day without
retraining. `ml.list_models()` lists every saved model from the tracked parameters, and
`saved_here` shows which are on this computer.

In [ ]:
# Switched off: select all and toggle comments (Ctrl+/) to run this cell.
# first = SAVED[('MLP', 'wdl', TICKERS[0])]
# loaded = ml.load_model(first.model_id) if SAVE_MODELS else first
# same = (loaded.forecasts.wdl.equals(first.forecasts.wdl)
#         and loaded.forecast_next().equals(first.forecast_next()))
# print(f'{loaded.model_id}: reloaded; forecasts identical to the trained model: {same}')
#
# wdl_next, ret_next = {}, {}
# for s in SAVED.values():
#     f = s.forecast_next()
#     key = (s.model.name, s.ticker)
#     if s.target == 'wdl':
#         wdl_next[key] = {'as of': f.name.date(), 'loss': f['loss'], 'draw': f['draw'], 'win': f['win']}
#     else:
#         close = BARS[s.ticker]['Close'].iloc[-1]
#         ret_next[key] = {'as of': f.name.date(), 'close': close, 'median': close * (1 + f[0.5]),
#                          '68% from': close * (1 + f[0.1587]), '68% to': close * (1 + f[0.8413])}
# display(pd.DataFrame(wdl_next).T.sort_index().style.format({'loss': '{:.1%}', 'draw': '{:.1%}', 'win': '{:.1%}'})
#         .set_caption('Next day: loss, draw and win probabilities'))
# display(pd.DataFrame(ret_next).T.sort_index().style.format({c: '{:,.2f}' for c in ('close', 'median', '68% from', '68% to')})
#         .set_caption('Next day: the close, its median forecast and the 68% range'))

## 10 The MLP against the saved LightGBM and GRUs

Nothing is retrained: the LightGBM and GRU forecasts are their saved versions' out-of-sample
forecasts (sections 4 and 5), the MLP's are this run's. On the days both forecast,
`ml.compare_forecasts(other, mlp)` gives the mean daily loss of each (log loss for win/draw/loss,
mean pinball for the return) and the gain of the MLP over the other model, with a 95% bootstrap
interval: `b_better` when it lies wholly above zero (the MLP is better), `a_better` when wholly
below. The chart shows the gain as a share of the other model's loss, so the two targets read on
the same footing; the table has every number.

In [ ]:
# Switched off: select all and toggle comments (Ctrl+/) to run this cell.
# OTHERS = {'LightGBM': (LGB_W, LGB_R), **{f'GRU ({i})': (GRU_W[i], GRU_R[i]) for i in GRU_INPUTS}}
# rows = []
# for name, (W, R) in OTHERS.items():
#     for target, theirs, ours in (('wdl', W, MLP_W), ('return', R, MLP_R)):
#         for t in LONG:
#             if t in theirs and t in ours:
#                 rows.append({**ml.compare_forecasts(theirs[t], ours[t]).to_dict(), 'ticker': t})
# COMPARE = pd.DataFrame(rows).set_index(['target', 'model_a', 'ticker'])
# display(COMPARE[['model_b', 'days', 'loss_a', 'loss_b', 'gain_b_over_a', 'lower', 'upper', 'b_better',
#                  'a_better']].style.format({'loss_a': '{:.6f}', 'loss_b': '{:.6f}', 'gain_b_over_a': '{:+.6f}',
#                                             'lower': '{:+.6f}', 'upper': '{:+.6f}'})
#         .set_caption('The MLP (b) against each saved model (a), per ticker'))
# ml.plot_model_comparison(COMPARE.reset_index()).show()
# COMPARE.groupby(['target', 'model_a']).agg(tickers=('days', 'size'), MLP_better=('b_better', 'sum'),
#                                            other_better=('a_better', 'sum'))